# 03 — Target Tracking, Distractor Resilience, and Recovery Analysis

This notebook evaluates session-scoped target tracking, distractor spatial association, ambiguity detection, loss timeout, expiry, and reacquisition recovery for stage 03.

It imports reusable evaluation functions from `kinetiq_v_vision.evaluation.tracking` and asserts that the three original scenarios (Clean Tracking, Distractor Crossing, Exit and Re-entry) remain zero-silent-target-switch. The scenario set also includes an adversarial "Bystander-Only Reassociation" scenario that reproduces a real, disclosed, not-yet-fixed defect: `TargetTracker`'s spatial fallback matching scores candidates by IoU/centroid distance only, with no identity check, so a same-position bystander can silently reach `CONFIRMED` under a different `candidate_id` once the true target leaves. This notebook is expected to report that defect, not hide it — VV-402 stays Blocked until the tracker itself is hardened. (The `bystander_attributed_reps` metric that used to appear here has been removed: repetition evaluation does not exist yet in this codebase, so that field was permanently and vacuously zero rather than measuring anything real.)

In [ ]:
from kinetiq_v_vision.evaluation.tracking import (
    create_synthetic_tracking_scenarios,
    run_tracking_evaluation,
)

print("Initializing Target Tracking Evaluation Stage 03...")

In [ ]:
scenarios = create_synthetic_tracking_scenarios()
print(f"Loaded {len(scenarios)} tracking evaluation scenarios:")
for s in scenarios:
    print(f" - {s.name}: {s.description} ({len(s.frames)} frames)")

In [ ]:
metrics = run_tracking_evaluation(scenarios)
print("\n--- Target Tracking Evaluation Summary ---")
print(f"Total Frames Evaluated:       {metrics.total_frames}")
print(f"Target Tracked Frames:        {metrics.target_tracked_frames}")
print(f"Attribution Accuracy:         {metrics.attribution_accuracy_pct}%")
print(f"Target Switch Errors:         {metrics.target_switch_count}")
print(f"False Pauses:                 {metrics.false_pause_count}")
print(f"Correct Pauses:               {metrics.correct_pause_count}")
print(f"Reacquisitions:               {metrics.reacquisition_count}")
print(f"Mean Reacquisition Latency:   {metrics.mean_reacquisition_latency_ms} ms")
print(f"Is Release Eligible:          {metrics.is_release_eligible}")
print(f"Is Synthetic Evaluation:      {metrics.is_synthetic}")

In [ ]:
# Programmatically enforce target tracking release invariants.
# The three original scenarios (Clean Tracking, Distractor Crossing, Exit
# and Re-entry) must remain switch-free -- any failure here is a real
# regression, not the known adversarial-scenario defect below.
clean_metrics = run_tracking_evaluation(scenarios[:3])
assert clean_metrics.target_switch_count == 0, (
    f"Regression: {clean_metrics.target_switch_count} silent target switch "
    "errors on scenarios previously verified switch-free!"
)
assert clean_metrics.is_release_eligible, (
    "Clean scenarios no longer meet release eligibility -- investigate before proceeding."
)
print("Clean scenarios (Clean Tracking, Distractor Crossing, Exit and Re-entry): switch-free, confirmed.")

# The adversarial "Bystander-Only Reassociation" scenario (included in
# `scenarios` above) is EXPECTED to be release-ineligible: it is a known,
# disclosed, not-yet-fixed defect in TargetTracker's spatial fallback
# matching (IoU/centroid distance only, no identity check), not a test bug.
# VV-402 stays Blocked until the tracker itself is hardened against this.
assert not metrics.is_release_eligible, (
    "Expected the full scenario set (including the adversarial scenario) to be "
    "release-ineligible; if this now passes, the tracker defect may have been "
    "fixed -- update this notebook and VV-402's roadmap state accordingly."
)
print(
    f"\nKNOWN, DISCLOSED DEFECT: {metrics.target_switch_count} silent target switches "
    "from the adversarial Bystander-Only Reassociation scenario. Target tracking "
    "evaluation correctly detects and reports this rather than claiming false "
    "release eligibility."
)